# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

**Method: Gradient-boosted trees (`HistGradientBoostingRegressor`), predicting `ctr_gap_last30`.**

Why this method for this lane:
- The signal audit (ML-06) found the CTR-gap signal is real but the relationship with position
  is non-linear and has heavy tails (a few high-impression pages dominate). Tree-based models
  handle non-linearity and outliers without manual transforms.
- ML-05's feature set mixes types that trees handle natively without much preprocessing: counts
  (`backlinks`), rates (`competition`), and categoricals with missing values (`_was_missing`
  flags already built in ML-05).
- It's still inspectable — feature importances give a plain-language "what is it leaning on"
  answer for Section 4, unlike a black-box deep model this dataset size doesn't need anyway.

**What the model is actually asked to do, precisely (resolving the open question from ML-03):**
predict `ctr_gap_last30` using *only* features available **before** the `last30` window —
`prev30` performance + static SEO/content metadata. It never sees this month's own
impressions/clicks/position, the same numbers the baseline (ML-07) uses directly. That keeps
the comparison in Section 3 honest: both the model and its baseline counterpart here are working
from the same, earlier information, predicting the same later outcome.


In [6]:
import pandas as pd
import numpy as np

from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import GroupKFold, KFold


# ============================================================
# 1. Load data
# ============================================================

DATA_DIR = "/home/mahad/projects/flyrank-ml-internship-starter/work/data"

fact = pd.read_parquet(
    f"{DATA_DIR}/fact_content_query_90d.parquet"
)

dc = pd.read_parquet(
    f"{DATA_DIR}/dim_content.parquet"
)


# ============================================================
# 2. Create target: ctr_gap_last30
# ============================================================

fact["ctr_last30"] = (
    fact["clicks_last30"] /
    fact["impressions_last30"].replace(0, np.nan)
)

bins = [0, 3, 5, 10, 20, 50, 1000]
pos_labels = ["1-3", "3-5", "5-10", "10-20", "20-50", "50+"]

fact["pos_bucket_last30"] = pd.cut(
    fact["avg_position_last30"],
    bins=bins,
    labels=pos_labels
).astype(str)

exp_last30 = fact.groupby(
    "pos_bucket_last30"
).apply(
    lambda g: g["clicks_last30"].sum() /
              g["impressions_last30"].sum()
)

fact["ctr_gap_last30"] = (
    fact["pos_bucket_last30"]
    .map(exp_last30)
    .astype(float)
    - fact["ctr_last30"]
)


# ============================================================
# 3. Define features
# ============================================================

FACT_FEATURES = [
    "query_char_count",
    "query_token_count",
    "impressions_prev30",
    "clicks_prev30",
    "avg_position_prev30",
    "content_visible_query_count",
    "rare_query_count",
    "rare_impressions_share"
]

DC_FEATURES = [
    "search_volume",
    "competition",
    "cpc",
    "backlinks",
    "char_count",
    "word_count"
]


# ============================================================
# 4. Keep live content
# ============================================================

dc_live = dc[
    dc["is_published"] & ~dc["is_deleted"]
][
    ["client_hash_id", "content_hash_id"] + DC_FEATURES
]


# ============================================================
# 5. Build modeling dataframe
# ============================================================

df = fact[
    [
        "client_hash_id",
        "content_hash_id",
        "query_hash_id",
        "pos_bucket_last30"
    ]
    + FACT_FEATURES
    + ["ctr_gap_last30"]
].merge(
    dc_live,
    on=["client_hash_id", "content_hash_id"],
    how="inner"
).dropna(
    subset=["ctr_gap_last30"]
)


# ============================================================
# 6. Create X, y, groups
# ============================================================

X = df[FACT_FEATURES + DC_FEATURES].copy()

for col in X.columns:
    if X[col].isnull().any():
        X[col + "_was_missing"] = X[col].isnull().astype(int)
        X[col] = X[col].fillna(X[col].median())

y = df["ctr_gap_last30"].copy()

groups = df["client_hash_id"]


print("X:", X.shape)
print("y:", y.shape)
print("unique clients:", groups.nunique())

X: (1814458, 21)
y: (1814458,)
unique clients: 49


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

**Grouped by `client_hash_id`, not random, not time-based (this data is a single fixed
snapshot, so there's no later time slice to split on).**

Random row-level splitting would put some queries from a given client in train and others from
the *same client* in test. A client's overall SEO practices (site structure, domain authority,
content strategy) are a hidden factor that correlates across all of that client's rows — a
random split would let the model partly "memorize" a client's baseline quality from train and
just apply it to that client's test rows, inflating the score without proving the model
generalizes to a client it's never seen. `GroupKFold` on `client_hash_id` forces every fold to
be evaluated on entirely unseen clients — an honest test of "does this work for a new client,"
which is the real use case (FlyRank's clients aren't in the training set when the product
ships).


In [7]:
gkf = GroupKFold(n_splits=5)
splits = list(gkf.split(X, y, groups=groups))
print("Folds:", len(splits))
for i, (tr, te) in enumerate(splits):
    print(f"Fold {i}: train clients={groups.iloc[tr].nunique()}, test clients={groups.iloc[te].nunique()}, "
          f"overlap={len(set(groups.iloc[tr]) & set(groups.iloc[te]))}")


Folds: 5
Fold 0: train clients=48, test clients=1, overlap=0
Fold 1: train clients=34, test clients=15, overlap=0
Fold 2: train clients=38, test clients=11, overlap=0
Fold 3: train clients=37, test clients=12, overlap=0
Fold 4: train clients=39, test clients=10, overlap=0


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

**Same data, same metric, same split as the baseline — the baseline is redefined slightly to
make that a fair fight (explained below), then both are scored the same way: Precision@50.**

ML-07's baseline computes `ctr_gap` from *this month's own* real numbers — it isn't predicting
anything, so it can't be compared apples-to-apples to a model that only sees `prev30`. For this
comparison, the baseline gets the same information handicap as the model: **`prev30_ctr_gap`**
— the identical rule (expected CTR for position bucket, minus actual CTR), computed on `prev30`
data instead of `last30`. Both the model and this baseline rank pages using only pre-`last30`
information; both get checked against the same real, already-known `ctr_gap_last30`.

**Precision@50:** of the top 50 pages each method ranks highest, what fraction land in the
*actual* top 50 by real `ctr_gap_last30`. Run the cell below and put the real two numbers here —
I have not written a winner in this paragraph on purpose.


In [8]:
# Baseline: the same rule, computed one window earlier (prev30), so it's not cheating with
# information the model doesn't get either.
fact["pos_bucket_prev30"] = pd.cut(fact.avg_position_prev30, bins=bins, labels=pos_labels).astype(str)
exp_prev30 = fact.groupby("pos_bucket_prev30").apply(
    lambda g: g.clicks_prev30.sum() / g.impressions_prev30.replace(0, np.nan).sum())
fact["ctr_prev30"] = fact.clicks_prev30 / fact.impressions_prev30.replace(0, np.nan)
fact["prev30_ctr_gap"] = fact.pos_bucket_prev30.map(exp_prev30).astype(float) - fact.ctr_prev30

# Bring prev30_ctr_gap onto df by the same join keys used everywhere else — no index-matching guesswork
df = df.merge(
    fact[["client_hash_id", "content_hash_id", "query_hash_id", "prev30_ctr_gap"]],
    on=["client_hash_id", "content_hash_id", "query_hash_id"], how="left"
)

def precision_at_k(scores, y_true, k=50):
    scores = pd.Series(scores).reset_index(drop=True)
    y_true = pd.Series(y_true).reset_index(drop=True)
    top_pred = set(scores.sort_values(ascending=False).head(k).index)
    top_true = set(y_true.sort_values(ascending=False).head(k).index)
    return len(top_pred & top_true) / k

model_precisions, baseline_precisions = [], []
for tr, te in splits:
    model = HistGradientBoostingRegressor(random_state=42)
    model.fit(X.iloc[tr], y.iloc[tr])
    pred = model.predict(X.iloc[te])
    y_te = y.iloc[te]
    baseline_te = df["prev30_ctr_gap"].iloc[te]

    model_precisions.append(precision_at_k(pred, y_te))
    baseline_precisions.append(precision_at_k(baseline_te, y_te))

results = pd.DataFrame({"fold": range(len(splits)),
                         "model_precision_at_50": model_precisions,
                         "baseline_precision_at_50": baseline_precisions})
print(results)
print()
print("Mean model Precision@50:", results.model_precision_at_50.mean())
print("Mean baseline Precision@50:", results.baseline_precision_at_50.mean())
print()

   fold  model_precision_at_50  baseline_precision_at_50
0     0                    0.0                      0.04
1     1                    0.0                      0.00
2     2                    0.0                      0.04
3     3                    0.0                      0.00
4     4                    0.0                      0.06

Mean model Precision@50: 0.0
Mean baseline Precision@50: 0.028000000000000004



## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

**Do not fill this in until the cell below has actually run — feature importances and error
buckets are exactly the kind of number that's easy to guess wrong.** What to look for once it
runs: (1) which features the model actually leans on — if it's dominated by `impressions_prev30`
alone, that's basically a fancier version of "big pages get flagged," worth saying plainly; (2)
whether errors are worse for a particular position bucket or a particular client — if the model
is much worse for small clients (fewer historical queries to learn from), that's a real
limitation to carry into ML-09 and the capstone, not something to bury.


In [9]:
import numpy as np
final_model = HistGradientBoostingRegressor(random_state=42)
final_model.fit(X, y)

try:
    importances = pd.Series(final_model.feature_importances_, index=X.columns).sort_values(ascending=False)
    print("Feature importances:")
    print(importances)
except AttributeError:
    print("HistGradientBoostingRegressor doesn't expose feature_importances_ directly — "
          "use sklearn.inspection.permutation_importance instead:")
    from sklearn.inspection import permutation_importance
    r = permutation_importance(final_model, X, y, n_repeats=5, random_state=42)
    print(pd.Series(r.importances_mean, index=X.columns).sort_values(ascending=False))

# Error by position bucket — is the model worse for high-position (competitive) content?
pred_all = final_model.predict(X)
err = pd.DataFrame({"pos_bucket": df["pos_bucket_last30"].values,
                     "abs_error": np.abs(pred_all - y.values)})
print()
print("Mean absolute error by position bucket:")
print(err.groupby("pos_bucket").abs_error.mean().sort_values())


HistGradientBoostingRegressor doesn't expose feature_importances_ directly — use sklearn.inspection.permutation_importance instead:
clicks_prev30                      0.072001
impressions_prev30                 0.052101
rare_impressions_share             0.009413
avg_position_prev30                0.008879
word_count                         0.006194
char_count                         0.005818
rare_query_count                   0.005587
content_visible_query_count        0.004473
query_char_count                   0.003269
competition                        0.001899
query_token_count                  0.000874
search_volume                      0.000689
search_volume_was_missing          0.000390
backlinks_was_missing              0.000386
char_count_was_missing             0.000378
cpc                                0.000213
backlinks                          0.000120
avg_position_prev30_was_missing    0.000034
competition_was_missing            0.000000
cpc_was_missing                 

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

In [11]:
print("ML-08 kernel test")
print("X exists:", "X" in globals())
print("y exists:", "y" in globals())
print("groups exists:", "groups" in globals())
print("X shape:", X.shape)

ML-08 kernel test
X exists: True
y exists: True
groups exists: True
X shape: (1814458, 21)
